# Neural Networks - Neurons

## 1. The Neuron

Why neuron? What inside it? How math works in it?

- One neuron receives info $X$, each of which has its importance, collected in $W$ -> $X \times W$ -> One score called **pre-activation** ($z$).
- Linear.

```math
z = WX + b
```

- Learn which info should matter.

> How aligned is the input with the pattern I'm looking for?

- For $WX$, weighted sum, dot product:
  - *Multiplication* scales importance.
  - *Addition* combines evidence.
- Bias $b$ works as a shift to compare $z$ to any threshold, without it, the weighted sum can only be compared to zero. Because activation functions evaluate whether this sum is positive or negative.

In [12]:
import numpy as np

class Neuron:

    def __init__(self, input_dim):
        self.weights = np.random.randn(input_dim)
        self.bias = np.random.randn()

    def forward(self, x):
        """
        Forward := data flow from input to output.
        """
        x = np.asarray(x)
        return np.dot(self.weights, x) + self.bias

In [2]:
np.random.seed(42)
neuron = Neuron(input_dim=3)
x = np.array([2.0, 4.0, 1.0])
z = neuron.forward(x)
print(z)

2.6110894958464446


Pytorch does the same thing.

In [7]:
import torch

layer = torch.nn.Linear(2, 1, bias=True)

with torch.no_grad():
    layer.weight[:] = torch.tensor([[2., 4.]])
    layer.bias[:] = torch.tensor([1.])

x = torch.tensor([[3., 5.]])
print("PyTorch:", layer(x))
print("NumPy:", np.dot([2, 4], [3, 5]) + 1)

PyTorch: tensor([[27.]], grad_fn=<AddmmBackward0>)
NumPy: 27


## 2. Decision Boundaries

- Hyperplanes separating space, each one divide space -> 2 halves.
- Each point can have $z = 0$, positive, or negative.
- A point on boundary, $z=0$, so it is defined by:

```math
W^TX + b = 0
```

For 2-dim input and weight:

```math
x = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix},
w = \begin{bmatrix} w_1 \\ w_2 \end{bmatrix} \\
\Rightarrow z = w_1x_1 + w_2x_2 + b = 0 \\
\Leftrightarrow x_2 = -\frac{w_1}{w_2}x_1 - \frac{b}{w_2}, w_2 \ne 0
```

That's a straight line.

- Plane for 3D, Hyperplane for XD.
- $w$ is perpendicular (normal) to decision boundary.
  - Every vector on boundary is orthogonal to $w$.
  - Moving along boundary doesn't change $z$.
  - Moving in direction of $w$ increases $z$ the fastest.

In 2D, if $w = [1, 0]^T$ -> neuron cares about moving left and right. If $w = [0, 1]^T$ -> neuron cares about moving up and down. If $w = [1, 1]^T$ -> neuron cares about moving diagonally.

- If no bias, boundary passes through the origin.
- Weights *rotate* the boundary, bias *slides* the boundary.
- No single neuron can represent **XOR**. ->
  - Stacking neurons
  - Non-linear.

In [ ]:
def score(x):
    pass

def predict(x):
    """
    Convert score of x into class with boundary z = 0.
    """
    return int(score(x) >= 0)

## 3. Activation Functions (Motivation)

Why activation functions? Why not use decision boundries?

- Without activation functions, every layer is a linear function of the prev ones, even linear of the input!!! Layers collapse to 1 layer.
- XOR problem, a linear hyperplane can't solve.
- Linearity -> Non-linearity.

With $z = W^Tx + b$:

```math
y = f(z) = f(W^Tx + b)
```

- While $z$ can rotate, scale, translate, $f$ now have to:
  - bend
  - squash
  - clip
  - curve
  - fold


> Activation bends the geometry of the feature space.

In [13]:
class Activation:
    """
    Base Activation.
    """
    ...

class Identity(Activation):
    
    def forward(self, x):
        return x

class Square(Activation):
    
    def forward(self, x):
        return x**2

## 4. The Complete Neuron

Nothing major. Check `Neuron` class.

## 5. Vectorising the Neuron

$X$ of size $(n, d)$

```math
X = \begin{bmatrix}
x_{11} & x_{12} & \cdots & x_{1d} \\
\vdots & \vdots & \cdots & \vdots \\
x_{n1} & x_{n2} & \cdots & x_{nd}
\end{bmatrix}
```

Similarly $W$ of size $(d, m)$.

Scalar $b$ of size $(1,)$, but we have **broadcasting**.

## 6. Numerical Stability

- Infinite decimal numbers.
- .1 + .2 = .3000000000000000004 because .1 cannot be represented exactly in binary.
- Accumulate error -> `nan` or `inf`.

**Floating-point rounding**

- float32 and float64
- More bit = slower on GPUs.

**Overflow**

- Number are larger than the # bits it can carry.
- `inf`.

**Underflow**

- e^-1000
- Lower than it can represent -> auto `= 0`

**Catastrophic cancellation**

- 1000000.123456 − 1000000.123455 = 0.000001.
- But computer only stores 1000000.12 then the result is 0.

### Stable Sigmoid

```math
\sigma(x) = \frac{1}{1 + e^{-x}}
```

if $x = -1000$, we have two cases:

```math
\sigma(x) = \begin{cases}
    \frac{1}{1 + e^{-x}}, & x \ge 0 \\
    \frac{e^x}{1 + e^x}, & x < 0
\end{cases}
```

### Stable Softmax

```math
\frac{x^{x_i}}{\sum_je^{xj}}
```

Suppose x = [1000, 999, 998] => $m = \max{x}$, then compute:

```math
\frac{e^{x_i - m}}{\sum_je^{x_j - m}}
```

Because $e^{-m}$ is cancelled out easily.

### Add $\epsilon$ to `log()`

To avoid $\log(0)$, we have $\log(x + \epsilon)$

### Torch

In [14]:
import torch

x = torch.tensor([1000., 999., 998.])
print(torch.softmax(x, dim=0))

tensor([0.6652, 0.2447, 0.0900])


## 7. Building a Production-Quality Neuron

- Scalability, Reuasablity
- Caching -> Backpropagation